# 01 — Baseline Models: Can a Machine Learn Price Elasticity?

## Goal
Train three models on our synthetic retail data and compare their ability
to predict daily units sold from price, inventory, and time features.

## Models
- Linear Regression (baseline — assumes linear relationships)
- Random Forest (captures nonlinearity, no assumptions)
- XGBoost (gradient boosting — industry standard for tabular data)

## What we expect to find
- Linear Regression will struggle — elasticity is nonlinear
- Random Forest and XGBoost should outperform it
- RMSE and MAE will tell us how wrong each model is on average

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

print("All libraries loaded ✅")

All libraries loaded ✅


## LOAD THE MODELLING TABLE

In [2]:
# Load the ML-ready modeling table we built in Phase 3
df = pd.read_csv("data/modeling_table.csv", parse_dates=["date"])

print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")
print(f"\nFirst 3 rows:")
df.head(3)

Shape: (262080, 24)

Columns:
['sku_id', 'date', 'units_sold', 'lag_7_units_sold', 'lag_14_units_sold', 'rolling_7d_avg', 'days_since_launch', 'inventory_ratio', 'discount_pct', 'days_to_season_end', 'season', 'is_holiday', 'markdown_stage', 'category_jackets', 'category_jeans', 'category_shoes', 'category_tshirts', 'dow_Friday', 'dow_Monday', 'dow_Saturday', 'dow_Sunday', 'dow_Thursday', 'dow_Tuesday', 'dow_Wednesday']

First 3 rows:


,sku_id,date,units_sold,lag_7_units_sold,lag_14_units_sold,rolling_7d_avg,days_since_launch,inventory_ratio,discount_pct,days_to_season_end,...,category_jeans,category_shoes,category_tshirts,dow_Friday,dow_Monday,dow_Saturday,dow_Sunday,dow_Thursday,dow_Tuesday,dow_Wednesday
0,SKU_0001,2022-01-01,8,NaN,NaN,NaN,0,0.982222,0.0,58,...,0,1,0,0,0,1,0,0,0,0
1,SKU_0001,2022-01-02,4,NaN,NaN,8.0,1,0.973333,0.0,57,...,0,1,0,0,0,0,1,0,0,0
2,SKU_0001,2022-01-03,5,NaN,NaN,6.0,2,0.962222,0.0,56,...,0,1,0,0,1,0,0,0,0,0


# We define features and targets

In [3]:
#what is the target? 

#what are we predicting
TARGET = "units_sold"



# Now we define the features

**This is where we drop identifiers (sku_id, date) and the target itself**

In [4]:
FEATURES = []

for col in df.columns:
    if col not in ['sku_id', 'date', TARGET]:
        FEATURES.append(col)

print(f"Target: {TARGET}")
print(f"\nNumber of features: {len(FEATURES)}")
print(f"\nFeatures:\n{FEATURES}")



Target: units_sold

Number of features: 21

Features:
['lag_7_units_sold', 'lag_14_units_sold', 'rolling_7d_avg', 'days_since_launch', 'inventory_ratio', 'discount_pct', 'days_to_season_end', 'season', 'is_holiday', 'markdown_stage', 'category_jackets', 'category_jeans', 'category_shoes', 'category_tshirts', 'dow_Friday', 'dow_Monday', 'dow_Saturday', 'dow_Sunday', 'dow_Thursday', 'dow_Tuesday', 'dow_Wednesday']


In [5]:
import duckdb
import pandas as pd

# Create an in-memory DuckDB connection
con = duckdb.connect(database=':memory:')

# Register the CSV as a DuckDB table (you can also use READ_CSV_AUTO directly)
con.execute("CREATE TABLE modeling_table AS SELECT * FROM read_csv_auto('data/modeling_table.csv')")

# Quick check from SQL
print(con.execute("SELECT COUNT(*) AS rows, COUNT(DISTINCT sku_id) AS skus FROM modeling_table").fetchdf())

     rows  skus
0  262080   500


**Now let us look at** 
# SEGMENT PERFORMANCE

### Cell: Avg units sold by markdown stage × popularity segment

Goal: show how average daily sales change across markdown stages (0=full price → 5=deepest discount) for each popularity segment (winner/normal/dead).  
What to look for: rising avg units as markdown_stage increases for winners/normals (elasticity), and a possible drop at stage 5 (markdown fatigue).

In [6]:
# Query 1: How does popularity segment affect sales and zeros?
# Type this out cell by cell:
query = """
SELECT
    s.popularity_segment,
    COUNT(*) AS total_rows,
    AVG(m.units_sold) AS avg_sales,
    SUM(CASE WHEN m.units_sold = 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS zero_sales_pct
FROM modeling_table m
JOIN read_csv_auto('data/sku_master.csv') s ON m.sku_id = s.sku_id
GROUP BY s.popularity_segment
ORDER BY avg_sales DESC
"""

print(con.execute(query).fetchdf())

  popularity_segment  total_rows  avg_sales  zero_sales_pct
0             winner       71751  11.898343       51.147719
1             normal      141729   4.208532       54.064447
2               dead       48600   1.322942       51.839506


### Cell: Avg units sold by markdown stage × popularity segment

Goal: show how average daily sales change across markdown stages (0=full price → 5=deepest discount) for each popularity segment (winner/normal/dead).  
What to look for:
- Elasticity: avg_units should generally increase with markdown_stage for winners and normals.
- Markdown fatigue: observe whether stage 5 drops compared to stage 4.
- Dead SKUs: expect little change across stages.

In [7]:
query = """
select
    m.markdown_stage,
    s.popularity_segment,
    round(AVG(m.units_sold), 3) as avg_units
from modeling_table m
join read_csv_auto('data/sku_master.csv') s ON m.sku_id = s.sku_id
group by m.markdown_stage, s.popularity_segment
order by m.markdown_stage, s.popularity_segment;
"""

print(con.execute(query).fetchdf())

    markdown_stage popularity_segment  avg_units
0                0               dead      1.338
1                0             normal      7.562
2                0             winner     17.980
3                1               dead      1.546
4                1             normal      8.308
5                1             winner     21.881
6                2               dead      1.560
7                2             normal      8.297
8                2             winner     22.736
9                3               dead      1.862
10               3             normal      9.125
11               3             winner     30.202
12               4               dead      2.143
13               4             normal      9.468
14               4             winner     31.859
15               5               dead      0.588
16               5             normal      1.461
17               5             winner      4.496


### Cell: Row counts by markdown_stage × popularity_segment

Goal: confirm sample sizes behind the previous averages.  
Why: very small counts at stage 5 would make the avg_units there noisy and less reliable.

In [8]:
query = """
SELECT
  m.markdown_stage,
  s.popularity_segment,
  COUNT(*) AS rows,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY m.markdown_stage), 2) AS pct_of_stage
FROM modeling_table m
JOIN read_csv_auto('data/sku_master.csv') s ON m.sku_id = s.sku_id
GROUP BY m.markdown_stage, s.popularity_segment
ORDER BY m.markdown_stage, s.popularity_segment;
"""
print(con.execute(query).fetchdf())

    markdown_stage popularity_segment   rows  pct_of_stage
0                0               dead   6888         30.91
1                0             normal  10227         45.89
2                0             winner   5170         23.20
3                1               dead   6781         30.17
4                1             normal  10831         48.19
5                1             winner   4864         21.64
6                2               dead   7134         30.07
7                2             normal  11501         48.47
8                2             winner   5093         21.46
9                3               dead   6198         27.49
10               3             normal  10922         48.45
11               3             winner   5424         24.06
12               4               dead   5928         26.33
13               4             normal  11308         50.22
14               4             winner   5281         23.45
15               5               dead  15671         10.

### Cell: Check how many SKU-days (and avg days-per-SKU) fall into each markdown_stage

Goal: confirm stage distribution and compute average days each SKU spends in each stage.  
Why: explain why stage 5 has so many rows (long stage duration vs. many SKUs).

In [9]:
query = """
WITH per_sku_stage AS (
  SELECT
    sku_id,
    markdown_stage,
    COUNT(*) AS sku_days_in_stage
  FROM modeling_table
  GROUP BY sku_id, markdown_stage
)
SELECT
  p.markdown_stage,
  SUM(p.sku_days_in_stage) AS total_rows_in_stage,
  ROUND(AVG(p.sku_days_in_stage), 2) AS avg_days_per_sku_in_stage,
  COUNT(*) AS num_skus_with_stage,
  ROUND(100.0 * SUM(p.sku_days_in_stage) / (SELECT COUNT(*) FROM modeling_table), 2) AS pct_of_all_rows
FROM per_sku_stage p
GROUP BY p.markdown_stage
ORDER BY p.markdown_stage;
"""
print(con.execute(query).fetchdf())

   markdown_stage  total_rows_in_stage  avg_days_per_sku_in_stage  \
0               0              22285.0                      44.57   
1               1              22476.0                      44.95   
2               2              23728.0                      47.46   
3               3              22544.0                      45.09   
4               4              22517.0                      45.03   
5               5             148530.0                     303.12   

   num_skus_with_stage  pct_of_all_rows  
0                  500             8.50  
1                  500             8.58  
2                  500             9.05  
3                  500             8.60  
4                  500             8.59  
5                  490            56.67  
